In [0]:
storage_account_name = ""
storage_account_key = ""
spark.conf.set(
f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
storage_account_key
)

In [0]:
# Load raw FD001 data
train_path = f"abfss://raw@{storage_account_name}.dfs.core.windows.net/FD001/train_FD001.txt"
test_path = f"abfss://raw@{storage_account_name}.dfs.core.windows.net/FD001/test_FD001.txt"

train_raw_df = spark.read.format("csv") \
    .option("sep", " ") \
    .option("header", "false") \
    .load(train_path)

test_raw_df = spark.read.format("csv") \
    .option("sep", " ") \
    .option("header", "false") \
    .load(test_path)

print(f"Train rows: {train_raw_df.count()}")
print(f"Test rows: {test_raw_df.count()}")
print(f"Columns detected: {len(train_raw_df.columns)}")
display(train_raw_df.limit(5))

Train rows: 20631
Test rows: 13096
Columns detected: 28


_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9,_c10,_c11,_c12,_c13,_c14,_c15,_c16,_c17,_c18,_c19,_c20,_c21,_c22,_c23,_c24,_c25,_c26,_c27
1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,2388.06,9046.19,1.30,47.47,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.00,39.06,23.4190,null,null
1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.30,47.49,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.00,39.00,23.4236,null,null
1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,21.61,554.26,2388.08,9052.94,1.30,47.27,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.00,38.95,23.3442,null,null
1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.30,47.13,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.00,38.88,23.3739,null,null
1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.00,2388.06,9055.15,1.30,47.28,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.00,38.90,23.4044,null,null


In [0]:
# C-MAPSS FD001 has 26 real columns + 2 empty trailing columns
columns = [
    "engine_id", "cycle",
    "op_setting_1", "op_setting_2", "op_setting_3",
    "sensor_1", "sensor_2", "sensor_3", "sensor_4", "sensor_5",
    "sensor_6", "sensor_7", "sensor_8", "sensor_9", "sensor_10",
    "sensor_11", "sensor_12", "sensor_13", "sensor_14", "sensor_15",
    "sensor_16", "sensor_17", "sensor_18", "sensor_19", "sensor_20",
    "sensor_21"
]

train_renamed = train_raw_df.toDF(*(columns + ["_extra1", "_extra2"])) \
    .drop("_extra1", "_extra2")

test_renamed = test_raw_df.toDF(*(columns + ["_extra1", "_extra2"])) \
    .drop("_extra1", "_extra2")

print("Columns renamed")
display(train_renamed.limit(5))

Columns renamed


engine_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,2388.06,9046.19,1.30,47.47,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.00,39.06,23.4190
1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.30,47.49,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.00,39.00,23.4236
1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,21.61,554.26,2388.08,9052.94,1.30,47.27,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.00,38.95,23.3442
1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.30,47.13,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.00,38.88,23.3739
1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.00,2388.06,9055.15,1.30,47.28,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.00,38.90,23.4044


In [0]:
from pyspark.sql.functions import col

# Cast engine_id and cycle to int
train_typed = train_renamed \
    .withColumn("engine_id", col("engine_id").cast("int")) \
    .withColumn("cycle", col("cycle").cast("int"))

test_typed = test_renamed \
    .withColumn("engine_id", col("engine_id").cast("int")) \
    .withColumn("cycle", col("cycle").cast("int"))

# Cast all sensor and op_setting columns to float
sensor_cols = [c for c in columns if c not in ["engine_id", "cycle"]]
for c in sensor_cols:
    train_typed = train_typed.withColumn(c, col(c).cast("float"))
    test_typed = test_typed.withColumn(c, col(c).cast("float"))

print("Types cast successfully")
train_typed.printSchema()

Types cast successfully
root
 |-- engine_id: integer (nullable = true)
 |-- cycle: integer (nullable = true)
 |-- op_setting_1: float (nullable = true)
 |-- op_setting_2: float (nullable = true)
 |-- op_setting_3: float (nullable = true)
 |-- sensor_1: float (nullable = true)
 |-- sensor_2: float (nullable = true)
 |-- sensor_3: float (nullable = true)
 |-- sensor_4: float (nullable = true)
 |-- sensor_5: float (nullable = true)
 |-- sensor_6: float (nullable = true)
 |-- sensor_7: float (nullable = true)
 |-- sensor_8: float (nullable = true)
 |-- sensor_9: float (nullable = true)
 |-- sensor_10: float (nullable = true)
 |-- sensor_11: float (nullable = true)
 |-- sensor_12: float (nullable = true)
 |-- sensor_13: float (nullable = true)
 |-- sensor_14: float (nullable = true)
 |-- sensor_15: float (nullable = true)
 |-- sensor_16: float (nullable = true)
 |-- sensor_17: float (nullable = true)
 |-- sensor_18: float (nullable = true)
 |-- sensor_19: float (nullable = true)
 |-- sensor

In [0]:
# Drop constant sensors (zero variance in FD001)
constant_sensors = [
    "sensor_1", "sensor_5", "sensor_6",
    "sensor_10", "sensor_16", "sensor_18", "sensor_19"
]

train_clean = train_typed.drop(*constant_sensors)
test_clean = test_typed.drop(*constant_sensors)

print("Constant sensors dropped")
print(f"Remaining columns: {train_clean.columns}")

Constant sensors dropped
Remaining columns: ['engine_id', 'cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


In [0]:
# Write to processed container (Silver layer)
train_silver_path = f"abfss://processed@{storage_account_name}.dfs.core.windows.net/FD001/train_silver"
test_silver_path = f"abfss://processed@{storage_account_name}.dfs.core.windows.net/FD001/test_silver"

train_clean.write.mode("overwrite").parquet(train_silver_path)
test_clean.write.mode("overwrite").parquet(test_silver_path)

print("Silver layer written successfully")
print(f"Train silver: {train_silver_path}")
print(f"Test silver: {test_silver_path}")

Silver layer written successfully
Train silver: abfss://processed@datalake60306249.dfs.core.windows.net/FD001/train_silver
Test silver: abfss://processed@datalake60306249.dfs.core.windows.net/FD001/test_silver


In [0]:
# Verify
train_check = spark.read.parquet(train_silver_path)
test_check = spark.read.parquet(test_silver_path)

print("=== SILVER LAYER VERIFICATION ===")
print(f"Train rows: {train_check.count()}")
print(f"Test rows: {test_check.count()}")
print(f"Columns: {len(train_check.columns)}")
train_check.printSchema()
display(train_check.limit(5))

=== SILVER LAYER VERIFICATION ===
Train rows: 20631
Test rows: 13096
Columns: 19
root
 |-- engine_id: integer (nullable = true)
 |-- cycle: integer (nullable = true)
 |-- op_setting_1: float (nullable = true)
 |-- op_setting_2: float (nullable = true)
 |-- op_setting_3: float (nullable = true)
 |-- sensor_2: float (nullable = true)
 |-- sensor_3: float (nullable = true)
 |-- sensor_4: float (nullable = true)
 |-- sensor_7: float (nullable = true)
 |-- sensor_8: float (nullable = true)
 |-- sensor_9: float (nullable = true)
 |-- sensor_11: float (nullable = true)
 |-- sensor_12: float (nullable = true)
 |-- sensor_13: float (nullable = true)
 |-- sensor_14: float (nullable = true)
 |-- sensor_15: float (nullable = true)
 |-- sensor_17: float (nullable = true)
 |-- sensor_20: float (nullable = true)
 |-- sensor_21: float (nullable = true)



engine_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21
1,1,-7.0E-4,-4.0E-4,100.0,641.82,1589.7,1400.6,554.36,2388.06,9046.19,47.47,521.66,2388.02,8138.62,8.4195,392.0,39.06,23.419
1,2,0.0019,-3.0E-4,100.0,642.15,1591.82,1403.14,553.75,2388.04,9044.07,47.49,522.28,2388.07,8131.49,8.4318,392.0,39.0,23.4236
1,3,-0.0043,3.0E-4,100.0,642.35,1587.99,1404.2,554.26,2388.08,9052.94,47.27,522.42,2388.03,8133.23,8.4178,390.0,38.95,23.3442
1,4,7.0E-4,0.0,100.0,642.35,1582.79,1401.87,554.45,2388.11,9049.48,47.13,522.86,2388.08,8133.83,8.3682,392.0,38.88,23.3739
1,5,-0.0019,-2.0E-4,100.0,642.37,1582.85,1406.22,554.0,2388.06,9055.15,47.28,522.19,2388.04,8133.8,8.4294,393.0,38.9,23.4044


In [0]:
# FINAL VERIFICATION
print("\n" + "="*60)
print("✅ NOTEBOOK 05 COMPLETE - SILVER LAYER READY")
print("="*60)
train_silver_verify = spark.read.parquet(train_silver_path)
test_silver_verify = spark.read.parquet(test_silver_path)
print(f"✅ Train Silver: {train_silver_verify.count()} rows, {len(train_silver_verify.columns)} columns")
print(f"✅ Test Silver: {test_silver_verify.count()} rows, {len(test_silver_verify.columns)} columns")
print(f"✅ Path: {train_silver_path}")
print("="*60)


✅ NOTEBOOK 05 COMPLETE - SILVER LAYER READY
✅ Train Silver: 20631 rows, 19 columns
✅ Test Silver: 13096 rows, 19 columns
✅ Path: abfss://processed@datalake60306249.dfs.core.windows.net/FD001/train_silver
